In [ ]:
from itertools import combinations,permutations,product,chain
from operator import itemgetter
from math import comb,factorial

"""
Signed Partial Matching between [n] and [m]
Is a map from subset of [n] to subset of [m] with sign +1 or -1
"""

def getz(I,i,d):
    if i < 0  or i >= len(I):
        return d
    else:
        return I[i]
    

def PM(m,n,k):
    assert(k <= min(n,m))
    #generate all subset of [n] with size k
    for domain in combinations(range(1,m+1), k):
        for codomain in permutations(range(1,n+1), k):
                yield (tuple(sorted(domain)),tuple(sorted(codomain)))

def basis(m,n,sigma):
    I , J = sigma
    A = []
    for a in range(1,m+1):
        m = 0
        for k in range(len(I)):
            if I[k] <= a:
                m = J[k]
        for b in range(1,m+1):
            A.append((a,b))
    return frozenset(A) 

def perpbasis(m,n,B):
    A = []
    for c in range(1,m+1):
        for d in range(1,n+1):
            if (c,d) not in B:
                A.append((d,c))
    return frozenset(A)
    


def Fourier(m,n,sigma):
    I,J = sigma
    if 1 in I:
        Q =  tuple(a+1 for a in J if a < n ) 
    else:
        Q = tuple(chain((1,), (a+1 for a in J if a < n )))
    if n in J:
        P =  tuple(a-1 for a in I if 1 < a) 
    else:
        P = tuple(chain((a-1 for a in I if 1<a ),(m,)))
    return (Q,P)


def APM(m,n):
    for k in range(0,min(m,n)+1):
        for sigma in PM(m,n,k):
            yield sigma



def PM2orbit(m,n,sigma):
    I,J = sigma
    assert(len(I)==len(J))
    P, Q = [],[]
    for i in range(len(I)):
        """
        Overall the space is 

        ...*x*********
        ...o**********
        ......*x******
        ......o*******
        .........**x**
        .........*x***
        .........o****
        ..............
        ..............

        
        ...*x******
        ...o*******
        ......*x***
        ......o****
        .........**
        .........*x
        .........o*
        ...........
        ...........

        where 
        (I[i], J[i]) indicate the corner of the * parts. 
        """
        a = getz(I, i + 1, m + 1) - I[i]
        b = J[i] - getz(J, i - 1, 0)
        #print(f"I[i] = {I[i]}, J[i] = {J[i]}, a={a}, b={b}")
        for k in range(min(a, b)):
            P.append(I[i] + k)
            Q.append(J[i] - k)
    return (tuple(P), tuple(Q))

def sizeallpm(m,n):
    """
    The size of all partial matching
    = sum_k |S_m/S_(m-k) x S_k| |S_n/S_(n-k)| 
    """
    return sum(comb(m,k)*comb(n,k)*factorial(k) for k in range(min(m,n)+1))

def testmap(m,n):
    print(f"Test ({m},{n})")
    A = set(APM(m,n))
    B = set(APM(n,m))
    #print(f"A: {A}")
    #print(f"B: {B}")
    print(f"All partial matching {sizeallpm(m,n)}")
    FA = set(Fourier(m,n,sigma) for sigma in A)
    #relevent partial matching
    RPM = set(PM2orbit(m,n,sigma) for sigma in APM(m,n))
    #print(RPM)
    print(f"PM {len(RPM)} = {len(A)}")
    for sigma in APM(m,n):

        fa = Fourier(m,n,sigma)
        print(f"Orig: {sigma} -> {PM2orbit(m,n,sigma)}")
        print(f"Four {fa} -> {PM2orbit(n,m,fa)}")


        fb = Fourier(n,m,fa)
        if fb != sigma:
            print(f"{sigma} -> {fa}->{fb}")
        Bsigma = basis(m,n,sigma)
        Psigma = perpbasis(m,n,Bsigma)
        Fbasis = basis(n,m,fa)
        if Fbasis != Psigma:
            print(f"{sigma} -> {fa}->{fb}")
            print(f"Basis: {Bsigma}")
    print(f"{len(A)},{len(B)}, {len(FA)}")
    print(f"B-FA:{B-FA}")
    print(f"FA-B:{FA-B}")





""""
 *
 * 
 *  -> ***...
 .
 .
 .


"""

In [ ]:
m, n, k = 6,7,2

testmap(m,n)